# Estudio y comparativa resultados distributed + neg out y high fixed

En este notebook vamos a analizar los resultados de la versión distribuida basada en complejidad que incluye la opción de ir quitando las variables con complejidad negativa (con un margen de tolerancia) y de ir fijando las que tienen un valor alto de complejidad. Este valor alto de complejidad es difícil de establecer porque siempre son valores bajitos y además varían en función de la medida de complejidad. Así, por ahora (y por temas computacionales) solo estamos ejecutando para kDN y fijándonos en sus valores. Además, debido a que es "más seguro" que las negativas son malas, damos más fuerza a esto. Es decir, si una variable toma valores negativos, se saca aunque previamente se hubiera fijado. Una vez tomas algún valor negativo estás totalmente fuera y no puedes volver a entrar. Pero si se fija la variable, sí puede volver a salir. Digamos que sacamos variables con más fuerza para ir limpiando.

En base a los resultados del notebook "Distributed_FS_Complexity_Analysis", ya solo aplicamos un método a la hora de obtener los resultados. Se ejecuta únicamente la opción "random choice". Luego el método es: tomar muestras bootstrap de variables e ir evaluando la complejidad siguiendo un esquema backward. La variable que sale en cada caso se escoge de manera aleatoria.

Aquí vamos tanto a estudiar los resultados de esta versión distribuida como a comparar su performance tanto en complejidad como en rendimiento con los métodos del SOTA. En todos los casos vamos a escoger el número k de variables como el número de variables informativas (sabemos cuál es porque estamos en el caso artificial). En nuestra versión distribuida hemos hecho un filtro previo de variables con correlación de Pearson superior a 0.9. Así, para que las comparaciones sean justas, hemos ejecutado los métodos del estado del arte tanto sin el filtro como con el filtro. Los métodos del SOTA con filtro tendrán un "_corr" en el nombre.

In [ ]:


# Función para generar datos sintéticos
def generate_synthetic_dataset(n_samples, n_informative, n_noise,n_redundant_linear, n_redundant_nonlinear,
                                flip_y, class_sep, n_clusters_per_class, weights, random_state=42, noise_std=0.05):
    rng = np.random.RandomState(random_state)

    # Generamos solo informativas + ruido
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_informative + n_noise,
        n_informative=n_informative,
        n_redundant=0,
        n_repeated=0,
        flip_y = flip_y,
        class_sep = class_sep,
        n_clusters_per_class = n_clusters_per_class,
        weights =weights,
        shuffle=False,
        random_state=random_state
    )

    # X = preprocessing.scale(X)
    df = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
    formulas = {}
    formulas_nonlinear = {}

    # Redundantes lineales
    for j in range(n_redundant_linear):
        idx1, idx2 = rng.choice(n_informative, size=2, replace=False)
        coef1, coef2 = rng.uniform(-2, 2, size=2)
        new_name = f"f{df.shape[1]}"
        new_feature = coef1*df[f"f{idx1}"] + coef2*df[f"f{idx2}"]
        if noise_std > 0:
            new_feature += rng.normal(0, noise_std, size=n_samples)
        df[new_name] = new_feature
        formulas[new_name] = f"{coef1:.2f}*f{idx1} + {coef2:.2f}*f{idx2}" + ("" if noise_std==0 else " + ruido")

    # Redundantes no lineales
    for j in range(n_redundant_nonlinear):
        idx = rng.choice(n_informative, size=2, replace=False)
        func = rng.choice([np.sin, np.cos, np.square, np.exp])
        new_name = f"f{df.shape[1]}"
        new_feature = func(df[f"f{idx[0]}"]) + df[f"f{idx[1]}"]
        if noise_std > 0:
            new_feature += rng.normal(0, noise_std, size=n_samples)
        df[new_name] = new_feature
        formulas_nonlinear[new_name] = f"{func.__name__}(f{idx[0]}) + f{idx[1]}" + ("" if noise_std==0 else " + ruido")

    dict_info_feature = {
        "informative": [f"f{i}" for i in range(n_informative)],
        "noise": [f"f{i}" for i in range(n_informative, n_informative + n_noise)],
        "redundant_linear": list(formulas.keys()),
        "redundant_nonlinear": list(formulas_nonlinear.keys()),
        "formulas_linear": formulas,
        "formulas_nonlinear": formulas_nonlinear
    }

    df[df.columns] = StandardScaler(with_mean=True, with_std=True).fit_transform(df)

    return df, y, dict_info_feature



def compute_gps(y_true, y_pred):
    """
    Calcula GPS para un problema binario.
    """
    cm = confusion_matrix(y_true, y_pred, labels=np.unique(y_true))

    TN, FP, FN, TP = cm.ravel()

    # métricas base
    PPV = TP / (TP + FP) if (TP + FP) > 0 else 0
    TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
    NPV = TN / (TN + FN) if (TN + FN) > 0 else 0
    TNR = TN / (TN + FP) if (TN + FP) > 0 else 0

    # F1+ y F1-
    F1_pos = 2 * (PPV * TPR) / (PPV + TPR) if (PPV + TPR) > 0 else 0
    F1_neg = 2 * (NPV * TNR) / (NPV + TNR) if (NPV + TNR) > 0 else 0

    # GPS
    GPS = 2 * (F1_pos * F1_neg) / (F1_pos + F1_neg) if (F1_pos + F1_neg) > 0 else 0
    return GPS




def plot_complexity_importances(df, dataset_name="Dataset", guided=True, save_path=None):
    """
    Dibuja:
    1) Importancias medias por variable y medida de complejidad
    2) Frecuencia de aparición (count_vars)

    guided: True si el dataset es guiado por complejidad, False si es random
    """
    sns.set(style="whitegrid", font_scale=1.1)

    title_prefix = "Guided by Complexity" if guided else "Random choice"

    # Reordenamos según la media de importancias (positivas = útiles puesto que quitarlas aumenta la complejidad)
    df_plot = df.copy()
    df_plot["mean_importance_norm"] = df_plot[["Hostility_importances_norm", "N1_importances_norm", "kDN_importances_norm"]].mean(axis=1)
    df_plot = df_plot.sort_values("mean_importance_norm", ascending=False)

    # Gráfico de importancias
    plt.figure(figsize=(10, 6))
    df_melt = df_plot.melt(value_vars=["Hostility_importances_norm", "N1_importances_norm", "kDN_importances_norm"],
                           var_name="Measure", value_name="Importance_norm", ignore_index=False).reset_index()

    sns.barplot(data=df_melt, x="index", y="Importance_norm", hue="Measure")
    plt.axhline(0, color="black", linewidth=1)
    plt.xticks(rotation=45)
    plt.xlabel("Variable")
    plt.ylabel("Mean importance (ΔComplexity)")
    plt.title(f"{dataset_name} — {title_prefix}\nPositive = variable reduces complexity")
    plt.legend(title="Complexity measure")
    plt.tight_layout()

    # if save_path:
    #     plt.savefig(f"{save_path}/{dataset_name}_importances_{'guided' if guided else 'random'}.png",
    #                 dpi=300, bbox_inches="tight")
    plt.show()

    # Gráfico de frecuencias
    plt.figure(figsize=(8, 4))
    sns.barplot(x=df_plot.index, y="count_vars", data=df_plot, color="skyblue")
    plt.xticks(rotation=45)
    plt.xlabel("Variable")
    plt.ylabel("Count")
    plt.title(f"{dataset_name} — Variable occurrence count ({title_prefix})")
    plt.tight_layout()

    # if save_path:
    #     plt.savefig(f"{save_path}/{dataset_name}_count_{'guided' if guided else 'random'}.png",
    #                 dpi=300, bbox_inches="tight")
    plt.show()



def compute_spearman_correlations(df, measures=["Hostility_importances_norm", "N1_importances_norm", "kDN_importances_norm"]):
    """
    Calcula la correlación de Spearman entre las medidas de complejidad
    ignorando NaN.
    Devuelve un DataFrame con la matriz de correlaciones.
    """
    # Filtramos solo las columnas relevantes y quitamos filas con NaN
    df_corr = df[measures].dropna(how="any")

    # Calculamos correlación
    corr, _ = spearmanr(df_corr)
    corr_matrix = pd.DataFrame(corr, index=measures, columns=measures)

    return corr_matrix








def analyze_negative_importances(csv_path, dict_info_feature):
    """
    Analiza las variables con importancia negativa en el métod distribuido.
    """
    df = pd.read_csv(csv_path, index_col=0)

    # Selección de medidas
    measures = ["Hostility_importances_norm", "N1_importances_norm", "kDN_importances_norm"]

    # Naturaleza variables en diccionario
    feature_types = {}
    for t, feats in dict_info_feature.items():
        for f in feats:
            feature_types[f] = t

    summary_list = []

    for m in measures:
        negatives = df[df[m] < 0][m]
        # if negatives.empty:
        #     continue

        # Clasificación de cada variable negativa
        neg_types = [feature_types.get(f, "unknown") for f in negatives.index]
        neg_df = pd.DataFrame({"feature": negatives.index, "importance": negatives.values, "type": neg_types})

        # Resumen por tipo
        type_counts = neg_df["type"].value_counts(normalize=True) * 100
        type_counts = type_counts.round(2)

        # Total negativas y proporciones
        summary_entry = {
            "measure": m,
            "n_negatives": len(neg_df),
            "pct_informative": type_counts.get("informative", 0),
            "pct_noise": type_counts.get("noise", 0),
            "pct_redundant_linear": type_counts.get("formulas_linear", 0),
            "pct_redundant_nonlinear": type_counts.get("formulas_nonlinear", 0),
        }

        summary_list.append(summary_entry)

    summary_df = pd.DataFrame(summary_list)
    return summary_df




###############################################################################################################
#####                          PLOT COMPLEXITY RESULTADOS VERSION CV                                      #####
###############################################################################################################
# Hacemos un plot tipo importancia de variables en RF

# df = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset2_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

def plot_complexity_importances_by_model(df, dataset_name="Dataset", save_path=None):
    """
    Produce para cada modelo:
    1) Gráfico de barras de kDN_importances_norm agregadas por feature (mean ± std)
    2) Gráfico de frecuencias (count_vars sumado)
    """

    sns.set(style="whitegrid", font_scale=1.1)
    modelos = df["model"].unique()
    n_models = len(modelos)
    modelo = modelos[0]

    # GRID: dos columnas (importancia, frecuencias)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes = np.array([axes])
    df_model = df[df["model"] == modelo]

    # agregamos por fold
    agg_df = (
        df_model.groupby("feature")
                    .agg(
                         kdn_mean=("kDN_importances_norm", "mean"),
                         kdn_median=("kDN_importances_norm", "median"),
                         kdn_std=("kDN_importances_norm", "std"),
                         count_total=("count_vars", "sum")
                    )
                    .sort_values("kdn_mean", ascending=False)
    )
    agg_df["kdn_std"] = agg_df["kdn_std"].fillna(0) # Evitar NaN en std cuando solo hay 1 fold

    # IMPORTANCIAS (mean ± std)
    ax1 = axes[0, 0]
    ax1.bar(
        agg_df.index,
        agg_df["kdn_mean"],
        yerr=agg_df["kdn_std"],
        capsize=4
    )
    ax1.axhline(0, color="black", linewidth=1)
    ax1.set_title(f"kDN importance (mean ± std)")
    ax1.set_xlabel("Variable")
    ax1.set_ylabel("Importance")
    ax1.tick_params(axis="x", rotation=45)

    # FRECUENCIA de aparición
    ax2 = axes[0, 1]
    ax2.bar(agg_df.index, agg_df["count_total"])
    ax2.set_title(f"Variable occurrence count")
    ax2.set_xlabel("Variable")
    ax2.set_ylabel("Count")
    ax2.tick_params(axis="x", rotation=45)

    plt.suptitle(f"{dataset_name} — Complexity Random FS", fontsize=16, y=1.02)
    plt.tight_layout()

    # if save_path:
    #     plt.savefig(f"{save_path}/{dataset_name}_Complexity_Grid.png",
    #                 dpi=300, bbox_inches="tight")

    plt.show()








############################################################################
######        PERFORMANCE POR FOLD SIGUIENDO RANKING out y high       ######
############################################################################
# Ya tenemos el ranking con kdn con neg out y high fixed por fold
# Ahora vamos a sacar la performance de los modelo según vamos añadiendo variables


# path_csv = 'Results_FS_Distributed_CV/ArtificialDataset12_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'

def load_importances_per_fold(path_csv, measures=["kDN_importances_norm"]):
    """
    Lee el archivo CSV con importancias por fold.
    Devuelve: dict[fold][measure] = DataFrame con columnas (feature, importance)
    Solo incluye las variables que tengan valor (no NaN).
    """
    df = pd.read_csv(path_csv)

    df = df.loc[df.model == 'KNN',:] # aquí los modelos no interactuan, realmente esa columna sobra en los csvs

    result = {}

    for fold in sorted(df["fold"].unique()):
        df_f = df[df["fold"] == fold]

        result[fold] = {}

        for m in measures:

            df_m = df_f[["feature", m]].dropna(subset=[m])  # elimina variables que quitamos por alta correlación

            # Orden descendente: mayor importancia primero
            df_m = df_m.sort_values(m, ascending=False)

            result[fold][m] = df_m.reset_index(drop=True)

    return result



# Función para evaluar rendimiento metiendo variables modo forward siguiendo el ranking
# establecido por la complejidad modo neg out high fixed
def evaluate_incremental_k(X, y, importances_dict, models, dataset_name, cv_splits=5, random_state=0):
    """
    Para cada fold, measure y modelo:
      evalúa rendimiento con k = 1..K variables ordenadas por importancia.
    """

    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)

    rows = []

    for fold_id, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        X_train_base, X_test_base = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        for measure, df_m in importances_dict[fold_id].items():

            features_ranked = df_m["feature"].tolist()
            K = len(features_ranked)

            for k in range(1, K + 1):

                selected = features_ranked[:k]

                Xt = X_train_base[selected]
                Xs = X_test_base[selected]

                for model_name, model in models.items():

                    clf = model
                    clf.fit(Xt, y_train)
                    pred = clf.predict(Xs)

                    acc = accuracy_score(y_test, pred)
                    gps = compute_gps(y_test, pred)

                    rows.append({
                        "dataset": dataset_name,
                        "fold": fold_id,
                        "measure": measure,
                        "k": k,
                        "n_available_features": K,
                        "model": model_name,
                        "acc_test": acc,
                        "gps_test": gps})

    perf_final = pd.DataFrame(rows)

    return perf_final



###3 Plot de performance evolutiva
def plot_incremental_performance(performance_df, importances_df, dataset, measure="acc_test"):
    """
    measure = 'acc_test' o 'gps_test'
    """

    df = performance_df.copy()
    df = df[df["dataset"] == dataset]
    models = df["model"].unique()
    folds = df["fold"].unique()

    # Colores por modelo
    colors = plt.cm.tab10(np.linspace(0, 1, len(models)))
    model_color = {m: c for m, c in zip(models, colors)}

    # Estilos por fold
    line_styles = ["solid", "dashed", "dotted", "dashdot"]
    fold_style = {f: line_styles[(f-1) % len(line_styles)] for f in folds}

    # mapa de negativo por fold
    negative_k = {}
    for fold in folds:
        imp_f = importances_df[importances_df["fold"] == fold]
        imp_sorted = imp_f.sort_values("kDN_importances_norm", ascending=False)
        neg_idx = np.where(imp_sorted["kDN_importances_norm"].values < 0)[0]
        negative_k[fold] = neg_idx[0] + 1 if len(neg_idx) > 0 else None

    plt.figure(figsize=(13, 8))

    for model in models:
        df_m = df[df["model"] == model]

        for fold in folds:
            df_f = df_m[df_m["fold"] == fold].sort_values("k")
            k = df_f["k"].values
            y = df_f[measure].values

            k_neg = negative_k[fold]

            # colores/estilos
            col = model_color[model]
            ls = fold_style[fold]

            # trozo principal (antes de k negativo)
            if k_neg is None:
                # Nada negativo: toda la curva normal
                plt.plot(k, y, marker="o", color=col, linestyle=ls,
                         alpha=0.8, label=f"{model} – fold {fold}")
            else:
                # tramo coloreado (hasta k_neg)
                idx_color = k <= k_neg
                plt.plot(k[idx_color], y[idx_color], marker="o",
                         color=col, linestyle=ls, alpha=0.8,
                         label=f"{model} – fold {fold}")

                # tramo gris
                idx_grey = k >= k_neg
                if np.any(idx_grey):
                    plt.plot(k[idx_grey], y[idx_grey], marker="o",
                             color="grey", linestyle=ls, alpha=0.6)

            # máximo
            max_row = df_f.loc[df_f[measure].idxmax()]
            plt.scatter(max_row["k"], max_row[measure],
                        s=80, edgecolor="black", facecolor="yellow", zorder=5)

            plt.text(max_row["k"], max_row[measure],
                     f"{max_row[measure]:.3f}",
                     fontsize=9, verticalalignment="bottom")

    plt.title(f"{dataset} – Evolución de {measure.replace('_',' ').upper()} con nº de variables")
    plt.xlabel("Número de variables usadas (k)")
    plt.ylabel(measure.replace("_"," ").upper())
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.legend()
    plt.show()






###########################################################################################
####                   ESTUDIO CARACTERÍSTICAS DE LAS VARIABLES                        ####
###########################################################################################
# Hacemos un script que caracterice el tipo de variable para ver cómo nos va quedando el ranking


# Función para extraer dependencias entre variables
def extract_dependencies(formula):
    """
    Extrae todas las variables f### que aparecen en la fórmula.
    """
    return re.findall(r"f\d+", formula)

# Diccionario  de dependeencias
def build_dependencies(dict_info_feature):
    deps = {}

    # Fórmulas lineales
    for var, formula in dict_info_feature["formulas_linear"].items():
        deps[var] = extract_dependencies(formula)

    # Fórmulas no lineales
    for var, formula in dict_info_feature["formulas_nonlinear"].items():
        deps[var] = extract_dependencies(formula)

    return deps

# ### Dataset 2
# dataset_name = 'ArtificialDataset2'
# X, y, dict_info_feature = generate_synthetic_dataset(n_samples=1000,n_informative=10,n_noise=2,
#                                          n_redundant_linear=4,n_redundant_nonlinear=2,
#                                     flip_y=0, class_sep = 0.6, n_clusters_per_class=1 , weights=[0.5],
#                                                      random_state=0,noise_std=0.01)
#
# dep2 = build_dependencies(dict_info_feature)


# poner tipo de cada variable en formato diccionario
def build_type_dict(dict_info_feature):
    type_dict = {}

    for v in dict_info_feature["informative"]:
        type_dict[v] = "informative"

    for v in dict_info_feature["noise"]:
        type_dict[v] = "noise"

    for v in dict_info_feature["redundant_linear"]:
        type_dict[v] = "redundant_linear"

    for v in dict_info_feature["redundant_nonlinear"]:
        type_dict[v] = "redundant_nonlinear"

    return type_dict

# type_dict2 = build_type_dict(dict_info_feature)



def caracterize_features_ranking(ranking, type_dict, dependencies):
    """
    ranking: lista ordenada de variables según importancia
    """
    selected = set()
    results = []

    for var in ranking:
        t = type_dict.get(var, "unknown")
        deps = dependencies.get(var, [])

        # Variable ruidosa
        if t == "noise":
            label = "noise"
        # lineal o no lineal
        elif t in ["redundant_linear", "redundant_nonlinear"]:
            if all(d in selected for d in deps) and len(deps) > 0:
                # Totalmente redundante
                label = t  # redundant_linear o redundant_nonlinear
            else:
                # Aún aporta algo
                label = "informative_derived"

        # Informativa original
        elif t == "informative":
            # PERO puede ser redundante
            # Si ya tenemos sus dependientes en selected
            redundant_now = False
            redundant_type = None
            for dep_var, dep_sources in dependencies.items(): # dep_var es la que depende de la informativa
                # si var participa en la formula
                if var in dep_sources and dep_var in selected: # la dependiente se ha seleccionado

                    # todas las otras fuentes necesarias para reconstruir var
                    other_sources = set(dep_sources) - {var}

                    # si TODAS ya están seleccionadas, la info de var está cubierta
                    if other_sources.issubset(selected):
                        redundant_now = True
                        redundant_type = type_dict.get(dep_var, None)
                        break

            if redundant_now:
                # convertimos informativa en redundante
                if redundant_type == "redundant_linear":
                    label = "informative_redundant_linear"
                else:
                    label = "informative_redundant_nonlinear"
            else:
                label = "informative"
        # no sabemos
        else:
            label = "unknown"

        results.append((var, label))
        selected.add(var)

    return results




# Con esta función contestamos a la pregunta: Cuál es la distribución de la tipología de variable
# de las k primeras variables siguiendo elranking?

def analyze_topk_with_labels(df_rankings, type_dict, dependencies, k_real, dataset_name="dataset"):
    """
    df_rankings: dataframe con columnas:
        - feature
        - kDN_importances_norm
        - fold

    type_dict: {feature: tipo_base}
    dependencies: {feature: [sources]}
    k_real: nº real de variables informativas
    """

    all_folds = sorted(df_rankings["fold"].unique())
    fold_results = []   # filas por fold
    all_labels = []     # para saber qué etiquetas existen

    for fold in all_folds:
        df_fold = df_rankings[df_rankings["fold"] == fold]
        df_sorted = df_fold.sort_values("kDN_importances_norm", ascending=False)

        ranking = df_sorted["feature"].tolist()
        labelled = caracterize_features_ranking(ranking, type_dict, dependencies)

        # guardar todas las labels para conocer el universo
        all_labels.extend([lab for _, lab in labelled])

        # LIMITAR AL TOP-k
        topk = labelled[:k_real]

        # contaje
        counts = {}
        for _, lab in topk:
            counts[lab] = counts.get(lab, 0) + 1

        # normalizar a porcentajes
        counts_pct = {f"pct_{lab}": counts.get(lab, 0) / k_real for lab in counts}

        counts_pct["fold"] = fold
        fold_results.append(counts_pct)

    # dataframe por fold
    df_folds = pd.DataFrame(fold_results).fillna(0)

    # labels globales
    unique_labels = sorted(set(all_labels))
    pct_cols = [c for c in df_folds.columns if c.startswith("pct_")]

    # resumen promedio por dataset
    summary = df_folds[pct_cols].mean().to_frame().T
    summary["dataset"] = dataset_name
    summary["k_real"] = k_real


    return summary, df_folds, unique_labels




def analyze_position_distribution(df_rankings, type_dict, dependencies):
    """
    Devuelve:
      - df_positions: DataFrame largo con columnas [feature, label, pos, fold]
      - pos_by_type: diccionario con listas de posiciones por tipo, agregando todos los folds
      - unique_labels: todas las etiquetas generadas por caracterize_features_ranking

    df_rankings: DataFrame con columnas:
        - feature
        - kDN_importances_norm
        - fold
    """

    all_folds = sorted(df_rankings["fold"].unique())
    records = []
    all_labels = []
    pos_by_type = {}

    for fold in all_folds:
        df_fold = df_rankings[df_rankings["fold"] == fold]
        df_sorted = df_fold.sort_values("kDN_importances_norm", ascending=False)

        ranking = df_sorted["feature"].tolist()

        # Clasificar usando tu función
        labelled = caracterize_features_ranking(ranking, type_dict, dependencies)

        for pos, (feat, label) in enumerate(labelled, start=1):
            records.append({
                "feature": feat,
                "label": label,
                "pos": pos,
                "fold": fold
            })
            all_labels.append(label)

            # Guardar posición por tipo
            if label not in pos_by_type:
                pos_by_type[label] = []
            pos_by_type[label].append(pos)

    df_positions = pd.DataFrame(records)
    unique_labels = sorted(set(all_labels))

    return df_positions, pos_by_type, unique_labels





In [ ]:
import copy
from sklearn import preprocessing
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from skrebate import ReliefF
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import glob
import re
#from DistributedFS_Complexity import * # lo comento porque da error no sé por qué

In [ ]:
import os
os.chdir("..")
root_path = os.getcwd()

In [ ]:
root_path

In [ ]:

def plot_complexity_importances_by_model(df, dataset_name="Dataset", save_path=None):
    """
    Produce para cada modelo:
    1) Gráfico de barras de kDN_importances_norm agregadas por feature (mean ± std)
    2) Gráfico de frecuencias (count_vars sumado)
    """

    sns.set(style="whitegrid", font_scale=1.1)
    modelos = df["model"].unique()
    n_models = len(modelos)
    modelo = modelos[0]

    # GRID: dos columnas (importancia, frecuencias)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes = np.array([axes])
    df_model = df[df["model"] == modelo]

    # agregamos por fold
    agg_df = (
        df_model.groupby("feature")
                    .agg(
                         kdn_mean=("kDN_importances_norm", "mean"),
                         kdn_median=("kDN_importances_norm", "median"),
                         kdn_std=("kDN_importances_norm", "std"),
                         count_total=("count_vars", "sum")
                    )
                    .sort_values("kdn_mean", ascending=False)
    )
    agg_df["kdn_std"] = agg_df["kdn_std"].fillna(0) # Evitar NaN en std cuando solo hay 1 fold

    # IMPORTANCIAS (mean ± std)
    ax1 = axes[0, 0]
    ax1.bar(
        agg_df.index,
        agg_df["kdn_mean"],
        yerr=agg_df["kdn_std"],
        capsize=4
    )
    ax1.axhline(0, color="black", linewidth=1)
    ax1.set_title(f"kDN importance (mean ± std)")
    ax1.set_xlabel("Variable")
    ax1.set_ylabel("Importance")
    ax1.tick_params(axis="x", rotation=45)

    # FRECUENCIA de aparición
    ax2 = axes[0, 1]
    ax2.bar(agg_df.index, agg_df["count_total"])
    ax2.set_title(f"Variable occurrence count")
    ax2.set_xlabel("Variable")
    ax2.set_ylabel("Count")
    ax2.tick_params(axis="x", rotation=45)

    plt.suptitle(f"{dataset_name} — Complexity Random FS", fontsize=16, y=1.02)
    plt.tight_layout()

    # if save_path:
    #     plt.savefig(f"{save_path}/{dataset_name}_Complexity_Grid.png",
    #                 dpi=300, bbox_inches="tight")

    plt.show()


### Resultados distributed

In [ ]:
# Dataset 2
df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset2_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

plot_complexity_importances_by_model(df_random, dataset_name="Dataset2")

In [ ]:
# Dataset  7
df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset7_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

plot_complexity_importances_by_model(df_random, dataset_name="Dataset7")

In [ ]:
# Dataset 12
df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset12_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

plot_complexity_importances_by_model(df_random, dataset_name="Dataset12")

In [ ]:
# # Dataset 14
df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset14_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

plot_complexity_importances_by_model(df_random, dataset_name="Dataset14")

In [ ]:
# # Dataset 18
df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset18_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

plot_complexity_importances_by_model(df_random, dataset_name="Dataset18")

In [ ]:
# # Dataset 20
df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset20_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

plot_complexity_importances_by_model(df_random, dataset_name="Dataset20")

In [ ]:
# # Dataset 21
# df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset21_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)
#
# plot_complexity_importances_by_model(df_random, dataset_name="Dataset21")

VER CUÁLES SON SIEMPRE POSITIVAS Y ANALIZAR CÓMO VA LA PERFORMANCE SOLO CON ELLAS, sacar cuántas son prositivas, corr entre modelos del ranking y relacionar el número de positivas con el número de informativas

## Tabla de comparación SOTA vs distributed (backward)

In [ ]:
# Leer todos los comparison.csv de la carpeta
files = glob.glob("Results_ComparisonDistributed_SOTA/*_ComparisonTable_CV_neg_high_DatasetVersions.csv")

all_tables = []
for f in files:
    df = pd.read_csv(f, index_col=[1])
    all_tables.append(df)

comparison_all = pd.concat(all_tables)
comparison_all.head()

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset2'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset7'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset12'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset14'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset18'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset20'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset21'])
# En este todavía no se ha ejecutado

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset18a'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset18b'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset18c'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset20a'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset20b'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset20c'])

## Performance evolutiva

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset2_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset2_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset2",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset2",measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset7_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset7_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset7",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset7", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset12_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset12_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset12",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset12", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset14_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset14_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset14",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset14", measure="gps_test")

En este caso se aprecia que la performance comienza a disminuir aunque no den valores negativos.

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset18_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset18_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset18",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset18", measure="gps_test")

En el dataset 18 vemos que la mayor parte de los máximos se logran con variables que ya aumentan la complejidad (zona gris). Se trata de un conjunto de datos bastante complejo.

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset18a_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset18a_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset18a",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset18a", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset18b_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset18b_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset18b",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset18b", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset18c_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset18c_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset18c",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset18c", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset20_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset20_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset20",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset20", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset20a_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset20a_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset20a",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset20a", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset20b_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset20b_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset20b",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset20b", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset20c_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset20c_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset20c",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset20c", measure="gps_test")

La SVM se comporta como esperamos pero el knn no. Su performance sigue aumentando hasta el final. Más adelante estudiamos cómo se distribuye la tipología de datos. Tampoco entiendo muy bien la diferencia de comportamiento entre los 2 modelos.

In [ ]:
# path_csv = 'Results_FS_Distributed_CV/ArtificialDataset21_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
# importances_dict = load_importances_per_fold(path_csv)
# importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)
#
# dfs = []
# for fold, v in importances_dict.items():
#     df = v["kDN_importances_norm"].copy()
#     df["fold"] = fold
#     dfs.append(df)
#
# importances_all = pd.concat(dfs, ignore_index=True)
# perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset21_OutHigh_EvolutivePerformance.csv')
#
# plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset21",measure="acc_test")

In [ ]:
# plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset21", measure="gps_test")

## Caracterización de variables

In [ ]:
dataset_name = 'ArtificialDataset2'
X, y, dict_info_feature = generate_synthetic_dataset(n_samples=1000,n_informative=10,n_noise=2,
                                         n_redundant_linear=4,n_redundant_nonlinear=2,
                                    flip_y=0, class_sep = 0.6, n_clusters_per_class=1 , weights=[0.5],
                                                     random_state=0,noise_std=0.01)
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset2_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)


ranking = importances_all.loc[importances_all.fold==1]
dep2 = build_dependencies(dict_info_feature)
type_dict2 = build_type_dict(dict_info_feature)
results = caracterize_features_ranking(ranking['feature'], type_dict2, dep2)


summary, df_folds, labels = analyze_topk_with_labels(
    df_rankings=importances_all,
    type_dict=type_dict2,
    dependencies=dep2,
    k_real=len(dict_info_feature['informative']))


In [ ]:
df_folds

Pilla bien las informativas

In [ ]:
df_positions, pos_by_type, labels = analyze_position_distribution(df_rankings=importances_all,
    type_dict=type_dict2,dependencies=dep2)


plt.figure(figsize=(10,5))
sns.boxplot(data=df_positions, x="label", y="pos", width=0.2, boxprops={'facecolor':'none'})
plt.title("Distribución de posiciones por tipo de variable")
plt.ylabel("Posición en el ranking (menor = mejor)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


In [ ]:
#### Dataset 7
dataset_name = 'ArtificialDataset7'
X, y, dict_info_feature = generate_synthetic_dataset(n_samples=1000,n_informative=20,n_noise=10,
                                         n_redundant_linear=10,n_redundant_nonlinear=10,
                                        flip_y=0, class_sep=1, n_clusters_per_class=1, weights=[0.5],
                                                     random_state=589,noise_std=0.05)

path_csv = 'Results_FS_Distributed_CV/ArtificialDataset7_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)


ranking = importances_all.loc[importances_all.fold==1]
dep2 = build_dependencies(dict_info_feature)
type_dict2 = build_type_dict(dict_info_feature)
results = caracterize_features_ranking(ranking['feature'], type_dict2, dep2)


summary, df_folds, labels = analyze_topk_with_labels(
    df_rankings=importances_all,
    type_dict=type_dict2,
    dependencies=dep2,
    k_real=len(dict_info_feature['informative']))

In [ ]:
df_folds

In [ ]:
df_positions, pos_by_type, labels = analyze_position_distribution(df_rankings=importances_all,
    type_dict=type_dict2,dependencies=dep2)


plt.figure(figsize=(10,5))
sns.boxplot(data=df_positions, x="label", y="pos", width=0.2, boxprops={'facecolor':'none'})
plt.title("Distribución de posiciones por tipo de variable")
plt.ylabel("Posición en el ranking (menor = mejor)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

Más o menos pilla bien las variables (el orden)

In [ ]:
#### Dataset 12
dataset_name = 'ArtificialDataset12'
X, y, dict_info_feature = generate_synthetic_dataset(n_samples=3000,n_informative=25,n_noise=30,
                                         n_redundant_linear=30,n_redundant_nonlinear=30,
                                        flip_y=0.2, class_sep=0.9, n_clusters_per_class=1, weights=[0.4],
                                                     random_state=987,noise_std=0.5)

path_csv = 'Results_FS_Distributed_CV/ArtificialDataset12_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)


ranking = importances_all.loc[importances_all.fold==1]
dep2 = build_dependencies(dict_info_feature)
type_dict2 = build_type_dict(dict_info_feature)
results = caracterize_features_ranking(ranking['feature'], type_dict2, dep2)


summary, df_folds, labels = analyze_topk_with_labels(
    df_rankings=importances_all,
    type_dict=type_dict2,
    dependencies=dep2,
    k_real=len(dict_info_feature['informative']))

In [ ]:
df_folds

In [ ]:
df_positions, pos_by_type, labels = analyze_position_distribution(df_rankings=importances_all,
    type_dict=type_dict2,dependencies=dep2)


plt.figure(figsize=(10,5))
sns.boxplot(data=df_positions, x="label", y="pos", width=0.2, boxprops={'facecolor':'none'})
plt.title("Distribución de posiciones por tipo de variable")
plt.ylabel("Posición en el ranking (menor = mejor)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

Se nos empiezan a colar más variablees redundants. No ruido, si no redundantes.

In [ ]:
#### Dataset 14
dataset_name = 'ArtificialDataset14'
X, y, dict_info_feature = generate_synthetic_dataset(n_samples=3000,n_informative=30,n_noise=40,
                                         n_redundant_linear=30,n_redundant_nonlinear=40,
                                        flip_y=0.2, class_sep=0.6, n_clusters_per_class=2, weights=[0.3],
                                                     random_state=95,noise_std=0.5)

path_csv = 'Results_FS_Distributed_CV/ArtificialDataset14_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)


ranking = importances_all.loc[importances_all.fold==1]
dep2 = build_dependencies(dict_info_feature)
type_dict2 = build_type_dict(dict_info_feature)
results = caracterize_features_ranking(ranking['feature'], type_dict2, dep2)


summary, df_folds, labels = analyze_topk_with_labels(
    df_rankings=importances_all,
    type_dict=type_dict2,
    dependencies=dep2,
    k_real=len(dict_info_feature['informative']))


In [ ]:
df_folds

In [ ]:
df_positions, pos_by_type, labels = analyze_position_distribution(df_rankings=importances_all,
    type_dict=type_dict2,dependencies=dep2)


plt.figure(figsize=(10,5))
sns.boxplot(data=df_positions, x="label", y="pos", width=0.2, boxprops={'facecolor':'none'})
plt.title("Distribución de posiciones por tipo de variable")
plt.ylabel("Posición en el ranking (menor = mejor)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

Se nos cuelan algunas redudantes (nada de ruido), pero generalmente bien.

In [ ]:
#### Dataset 18
dataset_name = 'ArtificialDataset18'
X, y, dict_info_feature = generate_synthetic_dataset(n_samples=500,n_informative=70,n_noise=40,
                                         n_redundant_linear=40,n_redundant_nonlinear=40,
                                        flip_y=0.4, class_sep=0.8, n_clusters_per_class=2, weights=[0.2],
                                                     random_state=9462,noise_std=0.5)

path_csv = 'Results_FS_Distributed_CV/ArtificialDataset18_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)


ranking = importances_all.loc[importances_all.fold==1]
dep2 = build_dependencies(dict_info_feature)
type_dict2 = build_type_dict(dict_info_feature)
results = caracterize_features_ranking(ranking['feature'], type_dict2, dep2)


summary, df_folds, labels = analyze_topk_with_labels(
    df_rankings=importances_all,
    type_dict=type_dict2,
    dependencies=dep2,
    k_real=len(dict_info_feature['informative']))

In [ ]:
df_folds

In [ ]:
df_positions, pos_by_type, labels = analyze_position_distribution(df_rankings=importances_all,
    type_dict=type_dict2,dependencies=dep2)


plt.figure(figsize=(10,5))
sns.boxplot(data=df_positions, x="label", y="pos", width=0.2, boxprops={'facecolor':'none'})
plt.title("Distribución de posiciones por tipo de variable")
plt.ylabel("Posición en el ranking (menor = mejor)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

El problema aquí es que se nos mete mucho ruido. Este dataset es difícil y es en el que más ruido se nos cuela.

In [ ]:
#### Dataset 20
dataset_name = 'ArtificialDataset20'
X, y, dict_info_feature = generate_synthetic_dataset(n_samples=500,n_informative=300,n_noise=60,
                                         n_redundant_linear=60,n_redundant_nonlinear=60,
                                        flip_y=0.1, class_sep=0.6, n_clusters_per_class=1, weights=[0.3],
                                                     random_state=4556,noise_std=0.5)


path_csv = 'Results_FS_Distributed_CV/ArtificialDataset20_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)


ranking = importances_all.loc[importances_all.fold==1]
dep2 = build_dependencies(dict_info_feature)
type_dict2 = build_type_dict(dict_info_feature)
results = caracterize_features_ranking(ranking['feature'], type_dict2, dep2)


summary, df_folds, labels = analyze_topk_with_labels(
    df_rankings=importances_all,
    type_dict=type_dict2,
    dependencies=dep2,
    k_real=len(dict_info_feature['informative']))

In [ ]:
df_folds

In [ ]:
df_positions, pos_by_type, labels = analyze_position_distribution(df_rankings=importances_all,
    type_dict=type_dict2,dependencies=dep2)


plt.figure(figsize=(10,5))
sns.boxplot(data=df_positions, x="label", y="pos", width=0.2, boxprops={'facecolor':'none'})
plt.title("Distribución de posiciones por tipo de variable")
plt.ylabel("Posición en el ranking (menor = mejor)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

Este es menos complejo y se cuela menos ruido, pero también se mete una cantidad importante.

PRÓXIMOS PASOS:
- Probar con distintos niveles de dificultad con la misma cantidad de variables para ver dónde está el problema.
- También poniendo distinta cantidad de variables informativas porque creo que poner 300 no es real.
- Hacer mini prueba con datos reales por si los estoy generando de forma rara.
- Ejecutar varias veces el conteo de variables para ver que es estable
- Verificar cómo se comportan métodos del SOTA en los datasets más complejos, quizás simplemente todo va mal ahí por las características de los datos y no tiene sentido intentar mejorar lo imposible.
- Recomendaciones de generación de datos para ML, más ceentrado en FS.

Hemos sacado más resultados con los datasets 18 y 20 que eran los que más dudas nos causaban. En particular, hemos sacado 3 versiones más de cada dataset: a, b y c. Estas versiones mantienen la misma estructura pero contienen menor número de variables. Los resultados que hemos observado son:
- Dataset 18. Este era muy complejo, mucho solapamiento, lío entre clases, etc. Al ir disminuyendo el número de variables, el accuracy se ha mantenido bastante estable y el GPS también. Lo único que ha ido cambiando han sido los mínimos de GPS que han ido aumentando un poco (0.15-0.25-0.3) (esto más o menos cuadra en el sentido de que hay menos variables y eso simplifica el problema). Los gráficos de performance evolutiva también revelan un comportamiento bastante errático en todos los casos. Así, a falta de estudiar los cambios en la performance evolutiva de los métodos del estado del arte, concluiría que el problema de este dataset no es el método nuestro si no la propia complejidad subyacente.
- Dataset 20. Este no era tan complejo (se lograban valores entornos a 0.8 de accuracy y GPS), pero la performance evolutiva mostraba un comportamiento raro. La SVM rbf funcionaba claramente peor que el knn. La SVM se quedaba como estancada y mostraba siempre una performance similar. Knn iba aumentando siempre su performance, un poco como "da igual lo que metas que siempre voy a mejorar). El comportamiento en las nuevas versiones a-b-c es un poco similar al caso del dataset 18. El accuracy se mantiene en niveles muy similares, los máximos de GPS también y lo que aumentan son los mínimos de GPS que pasan de 0.5 a 0.7 (en el caso c). Además, los gráficos de performance evolutiva muestran que la SVM ya sí pilla mejor la estructura y en general se obtiene el patrón deseado: cuando los puntos empiezan a ser grises (esas variables aumentan la complejidad), la performance se mantiene estable o disminuye. Esto no se ve en el caso del dataset 20, solo en el de sus versiones a-b-c. Falta por estudiar cómo se comportan los métodos del SOTA para sacar conclusiones claras porque sí parece que aquí el número de variables es muy determinante para la SVM.

Vamos a estudiar ahora el comportamieento de performance evolutivo de los datasets con los métodos del SOTA.

In [ ]:
def plot_incremental_performance_sota(performance_df,dataset,method,measure="acc_test"):
    """
    Gráfico incremental de rendimiento para métodos SOTA.
    """

    df = performance_df.copy()
    df = df[(df["dataset"] == dataset) & (df["method"] == method)]

    models = sorted(df["model"].unique())
    folds = sorted(df["fold"].unique())

    # Colores por modelo
    model_color = {"SVM-rbf": plt.cm.tab10(0),"KNN": plt.cm.tab10(9)}

    # Estilos por fold
    line_styles = ["solid", "dashed", "dotted", "dashdot"]
    fold_style = {f: line_styles[(i % len(line_styles))]for i, f in enumerate(folds) }

    plt.figure(figsize=(13, 8))

    for model in models:
        df_m = df[df["model"] == model]

        for fold in folds:
            df_f = df_m[df_m["fold"] == fold].sort_values("k")

            k = df_f["k"].values
            y = df_f[measure].values

            plt.plot(k, y,marker="o",color=model_color.get(model, "grey"),
                linestyle=fold_style[fold],alpha=0.85,label=f"{model} – fold {fold}")

            # marcar máximo por (modelo, fold)
            idx_max = df_f[measure].idxmax()
            max_row = df_f.loc[idx_max]

            plt.scatter(max_row["k"],max_row[measure],s=80,edgecolor="black",facecolor="yellow",
                zorder=5)

            plt.text(max_row["k"],max_row[measure],f"{max_row[measure]:.3f}",
                fontsize=9,verticalalignment="bottom")

    plt.title(f"{dataset} – {method} – Evolución de {measure.replace('_',' ').upper()}")
    plt.xlabel("Número de variables usadas (k)")
    plt.ylabel(measure.replace("_", " ").upper())
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.legend()
    plt.show()


In [ ]:
path_csv = 'Results_FS_SOTA_CV/ArtificialDataset12_SOTA_EvolutivePerformance.csv'
df_12 = pd.read_csv(path_csv)
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_12,dataset="ArtificialDataset12",
        method=method, measure="acc_test")

In [ ]:
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_12,dataset="ArtificialDataset12",
        method=method, measure="gps_test")

In [ ]:
path_csv = 'Results_FS_SOTA_CV/ArtificialDataset14_SOTA_EvolutivePerformance.csv'
df_14 = pd.read_csv(path_csv)
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_14,dataset="ArtificialDataset14",
        method=method, measure="acc_test")

In [ ]:
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_14,dataset="ArtificialDataset14",
        method=method, measure="gps_test")

In [ ]:
path_csv = 'Results_FS_SOTA_CV/ArtificialDataset18_SOTA_EvolutivePerformance.csv'
df_18 = pd.read_csv(path_csv)
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_18,dataset="ArtificialDataset18",
        method=method, measure="acc_test")

In [ ]:
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_18,dataset="ArtificialDataset18",
        method=method, measure="gps_test")

In [ ]:
path_csv = 'Results_FS_SOTA_CV/ArtificialDataset18a_SOTA_EvolutivePerformance.csv'
df_18a = pd.read_csv(path_csv)
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_18a,dataset="ArtificialDataset18a",
        method=method, measure="acc_test")

In [ ]:
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_18a,dataset="ArtificialDataset18a",
        method=method, measure="gps_test")

In [ ]:
path_csv = 'Results_FS_SOTA_CV/ArtificialDataset18b_SOTA_EvolutivePerformance.csv'
df_18b = pd.read_csv(path_csv)
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_18b,dataset="ArtificialDataset18b",
        method=method, measure="acc_test")

In [ ]:
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_18b,dataset="ArtificialDataset18b",
        method=method, measure="gps_test")

In [ ]:
path_csv = 'Results_FS_SOTA_CV/ArtificialDataset18c_SOTA_EvolutivePerformance.csv'
df_18c = pd.read_csv(path_csv)
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_18c,dataset="ArtificialDataset18c",
        method=method, measure="acc_test")

In [ ]:
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_18c,dataset="ArtificialDataset18c",
        method=method, measure="gps_test")

In [ ]:
path_csv = 'Results_FS_SOTA_CV/ArtificialDataset20_SOTA_EvolutivePerformance.csv'
df_20 = pd.read_csv(path_csv)
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_20,dataset="ArtificialDataset20",
        method=method, measure="acc_test")

In [ ]:
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_20,dataset="ArtificialDataset20",
        method=method, measure="gps_test")

In [ ]:
path_csv = 'Results_FS_SOTA_CV/ArtificialDataset20a_SOTA_EvolutivePerformance.csv'
df_20a = pd.read_csv(path_csv)
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_20a,dataset="ArtificialDataset20a",
        method=method, measure="acc_test")

In [ ]:
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_20a,dataset="ArtificialDataset20a",
        method=method, measure="gps_test")

In [ ]:
path_csv = 'Results_FS_SOTA_CV/ArtificialDataset20b_SOTA_EvolutivePerformance.csv'
df_20b = pd.read_csv(path_csv)
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_20b,dataset="ArtificialDataset20b",
        method=method, measure="acc_test")

In [ ]:
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_20b,dataset="ArtificialDataset20b",
        method=method, measure="gps_test")

In [ ]:
path_csv = 'Results_FS_SOTA_CV/ArtificialDataset20c_SOTA_EvolutivePerformance.csv'
df_20c = pd.read_csv(path_csv)
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_20c,dataset="ArtificialDataset20c",
        method=method, measure="acc_test")

In [ ]:
for method in ["f_classif", "mutual_info", "relief", "rf"]:
    plot_incremental_performance_sota(performance_df=df_20c,dataset="ArtificialDataset20c",
        method=method, measure="gps_test")

El análisis de estos resultados y su comparación con nuestro método está en el archivo: Comparacion_Artificial_SOTA.ods. Pero básicamente es:
* Con respecto a patrón evolutivo, nuestro método, relief y RF son los que más se parecen y los que revelan el comportamiento más “esperado”, esto es, un crecimiento inicial fuerte y claro, una estabilidad llegado el número de variables informativas y un descenso en el rendimiento según nos vamos alejando de esa zona de equilibrio. Quizás diría que en nuestro método es donde mejor se ve ese descenso pero creo que está sesgado por el cambio de color. Mutual info y f_classif (anova) son los que muestran peor patrón.
* Con respecto a patrón general de “pillar” los datasets, todos se comportan similar. Es decir, no es que nuestro método no pille ciertos datasets, es que son difíciles y les resultan difíciles a todos los métodos. También vemos que en aquellos más complejos, todo es un poco caótico pero, según se va disminuyendo el número de variables (y se mantiene fija la complejidad) vemos que el rendimiento puede mejorar un poco. Evidentemente esto se estanca puesto que la mejora está topada por la complejidad propia del dataset debido a solapamiento, etc., pero es verdad que quitar un poco de ruido de muchas variables pues da un mejora leve.
* Tengo que estudiar si los métodos del estado del arte de FILTRO dan alguna idea sobre cuántas variables elegir, porque creo que no y eso es un punto nuestro a favor.
